# 05 – Bronze: incremental load from PostgreSQL into Iceberg tables

**Goal (Task 3, parts 2 and 3):**

1. Upgrade the Task 2 **incremental** load (`append` on `updated_at`) so it writes **Iceberg tables** registered in the catalog, in the **bronze** zone, **partitioned by ingestion date**.
2. Read the tables with **PyIceberg** and verify the data.
3. Add records in the app, rerun, and use **snapshots** to read the table **before and after** (time travel).
4. Bonus: the same incremental load into the additional pile (RustFS).

What changed compared to notebook 02:

| | Task 2 (notebook 02) | Task 3 (this notebook) |
|---|---|---|
| destination | `lake_destination("silo")` → bucket `pawhome-lake` | `pile_destination("silo")` → bucket `pawhome-pile` |
| format | Parquet files | **Iceberg tables** (`table_format="iceberg"`) |
| table | folder `postgres/animals_incremental/` | `bronze.postgres_animals` in the **catalog** |
| partition | in the key (`ingestion_date=` in the layout) | **Iceberg partition** `ingestion_date = day(_ingested_at)` |

In [1]:
import dlt
import pandas as pd
import sqlalchemy as sa
from dlt.sources.sql_database import sql_table
from lake import POSTGRES_URL, DB_TABLES, PIPELINES_DIR, add_metadata
from pile import (BRONZE, get_catalog, pile_destination, partition_by_ingestion_date, run_to_pile,
                  ensure_pile_bucket, list_pile, show_catalog_db, snapshots, data_files)

pd.set_option("display.max_colwidth", 120)
engine = sa.create_engine(POSTGRES_URL)
catalog = get_catalog("silo")

## Step 1 – The resources

Same as notebook 02 (incremental on `updated_at` + metadata columns), plus:
- table name **`postgres_<table>`** → the source system is in the table name: `bronze.postgres_animals`
- **`partition_by_ingestion_date`** → Iceberg partition on the day of `_ingested_at`

In [2]:
def build_bronze_resources():
    resources = []
    for name in DB_TABLES:
        r = sql_table(POSTGRES_URL, table=name,
                      incremental=dlt.sources.incremental("updated_at"))
        r.apply_hints(table_name=f"postgres_{name}")
        add_metadata(r, source_system="postgres", source_resource=name, ingestion_type="incremental")
        partition_by_ingestion_date(r)
        resources.append(r)
    return resources


bronze_pipeline = dlt.pipeline(
    pipeline_name="pile_incremental",
    destination=pile_destination("silo"),
    dataset_name=BRONZE,                 # dataset = namespace = medallion zone
    pipelines_dir=PIPELINES_DIR,
)

## Step 2 – First run (loads everything)

`run_to_pile(...)` = `pipeline.run(..., write_disposition="append", table_format="iceberg", loader_file_format="parquet")`, after telling dlt to use **our SQL catalog**.

In [3]:
run_to_pile(bronze_pipeline, build_bronze_resources(), write_disposition="append")

/opt/conda/lib/python3.13/site-packages/pyiceberg/io/__init__.py:380: UserWarning: No preferred file implementation for scheme: memory
  if file_io := _infer_file_io_from_scheme(location, properties):


[silo] new rows loaded: {'postgres_adopters': 2000, 'postgres_animals': 3003, 'postgres_applications': 5000, 'postgres_shelters': 50}


### The tables are registered in the catalog

In [4]:
print("Tables in bronze:", catalog.list_tables(BRONZE))
show_catalog_db("silo")

Tables in bronze: [('bronze', 'postgres_adopters'), ('bronze', 'postgres_animals'), ('bronze', 'postgres_applications'), ('bronze', 'postgres_shelters')]


,table_namespace,table_name,metadata_location
0,bronze,postgres_adopters,s3://pawhome-pile/bronze/postgres_adopters/metadata/00001-7d470e6a-0eae-4d6a-a08d-21608bb399e8.metadata.json
1,bronze,postgres_animals,s3://pawhome-pile/bronze/postgres_animals/metadata/00001-4f71ac13-84ec-4fc8-a934-d5c66369303b.metadata.json
2,bronze,postgres_applications,s3://pawhome-pile/bronze/postgres_applications/metadata/00001-a869d376-1781-47e9-b80f-70ac669a3b33.metadata.json
3,bronze,postgres_shelters,s3://pawhome-pile/bronze/postgres_shelters/metadata/00001-832152c4-e70e-426b-a37e-681fc2b8834d.metadata.json


### The object keys

`bronze/<source>_<table>/data/ingestion_date=<YYYY-MM-DD>/<file>.parquet` = the data
`bronze/<source>_<table>/metadata/...` = Iceberg metadata (JSON = table description, `snap-*.avro` = snapshot, `*-m0.avro` = list of data files)

In [5]:
list_pile("bronze/postgres_animals")

,key,size_kb,last_modified
0,bronze/postgres_animals/data/ingestion_date=2026-11-06/00000-0-69013777-0004-4fdb-b163-a6ed6d6692db.parquet,108.5,2026-11-06 10:59:15.783+00:00
1,bronze/postgres_animals/metadata/00000-37c2a6a1-24e3-45e2-be03-969eff2015a6.metadata.json,1.8,2026-11-06 10:59:15.334+00:00
2,bronze/postgres_animals/metadata/00001-4f71ac13-84ec-4fc8-a934-d5c66369303b.metadata.json,2.7,2026-11-06 10:59:15.932+00:00
3,bronze/postgres_animals/metadata/69013777-0004-4fdb-b163-a6ed6d6692db-m0.avro,5.9,2026-11-06 10:59:15.819+00:00
4,bronze/postgres_animals/metadata/snap-1520607404253395009-0-69013777-0004-4fdb-b163-a6ed6d6692db.avro,1.8,2026-11-06 10:59:15.851+00:00


## Step 3 – Read with PyIceberg and verify

For every table: rows in the Iceberg table vs. rows in PostgreSQL, and the same ids.

In [6]:
checks = []
for name in DB_TABLES:
    table = catalog.load_table(f"{BRONZE}.postgres_{name}")
    df = table.scan().to_pandas()
    db = pd.read_sql(f"SELECT id FROM {name}", engine)
    checks.append({
        "iceberg_table": f"{BRONZE}.postgres_{name}",
        "rows_iceberg": len(df),
        "rows_postgres": len(db),
        "same_ids": set(df.id) == set(db.id),
        "partition": str(table.spec()),
    })
pd.DataFrame(checks)

,iceberg_table,rows_iceberg,rows_postgres,same_ids,partition
0,bronze.postgres_shelters,50,50,True,[\n 1000: ingestion_date: day(13)\n]
1,bronze.postgres_adopters,2000,2000,True,[\n 1000: ingestion_date: day(14)\n]
2,bronze.postgres_animals,3003,3003,True,[\n 1000: ingestion_date: day(17)\n]
3,bronze.postgres_applications,5000,5000,True,[\n 1000: ingestion_date: day(12)\n]


In [7]:
animals = catalog.load_table(f"{BRONZE}.postgres_animals")
print(animals.schema())

table {
  1: id: required long
  2: shelter_id: required long
  3: name: required string
  4: species: required string
  5: breed: optional string
  6: sex: optional string
  7: age_years: optional decimal(4, 1)
  8: size: optional string
  9: is_vaccinated: required boolean
  10: status: required string
  11: arrived_at: required date
  12: description: optional string
  13: updated_at: required timestamp
  14: _source_system: optional string
  15: _source_resource: optional string
  16: _ingestion_type: optional string
  17: _ingested_at: optional timestamptz
  18: _load_id: optional string
  19: _dlt_load_id: required string
  20: _dlt_id: required string
}


In [8]:
# PyIceberg can filter and select columns while reading
animals.scan(row_filter="species == 'dog'",
             selected_fields=("id", "name", "breed", "updated_at",
                              "_source_system", "_ingestion_type", "_ingested_at", "_load_id"),
             limit=5).to_pandas()

,id,name,breed,updated_at,_source_system,_ingestion_type,_ingested_at,_load_id
0,16,Oreo,Coton de Tulear,2025-01-13 00:00:16,postgres,incremental,2026-11-06 10:59:09.265342+00:00,1793962749.9484987
1,17,Daisy,Drentse Patrijshond,2026-02-12 00:00:17,postgres,incremental,2026-11-06 10:59:09.265342+00:00,1793962749.9484987
2,18,Oreo,Boerboel,2025-10-29 00:00:18,postgres,incremental,2026-11-06 10:59:09.265342+00:00,1793962749.9484987
3,19,Jax,Shar Pei,2025-08-17 00:00:19,postgres,incremental,2026-11-06 10:59:09.265342+00:00,1793962749.9484987
4,26,Archie,Smalandstövare,2024-10-24 00:00:26,postgres,incremental,2026-11-06 10:59:09.265342+00:00,1793962749.9484987


## Step 4 – Rerun without changes

Nothing is new → no rows, **no new snapshot**.

In [9]:
run_to_pile(bronze_pipeline, build_bronze_resources(), write_disposition="append")
snapshots(catalog.load_table(f"{BRONZE}.postgres_animals"))

[silo] new rows loaded: none


,snapshot_id,committed_at,operation,added_rows,deleted_rows,total_rows,schema_id
0,1520607404253395009,2026-11-06 10:59:15.861+00:00,append,3003,0,3003,0


## Step 5 – Time travel

First remember the **current snapshot** (= the version "before").

In [10]:
animals = catalog.load_table(f"{BRONZE}.postgres_animals")
snapshot_before = animals.current_snapshot().snapshot_id
print("Snapshot BEFORE:", snapshot_before)

Snapshot BEFORE: 1520607404253395009


**Now add some animals.**

- **Option A (best for the presentation):** open the app http://localhost:5173 → Animals → New, add 2–3 animals, then **skip** the next cell.
- **Option B:** run the next cell, it inserts 3 animals with SQL.

In [11]:
# Option B – skip this cell if you added animals in the app
with engine.begin() as conn:
    new_ids = conn.execute(sa.text("""
        INSERT INTO animals (shelter_id, name, species, breed, sex, age_years, size, arrived_at)
        VALUES (1, 'Pile Test 1', 'dog', 'Beagle', 'M', 2, 'medium', CURRENT_DATE),
               (1, 'Pile Test 2', 'cat', 'Persian', 'F', 1, 'small', CURRENT_DATE),
               (2, 'Pile Test 3', 'rabbit', 'Holland Lop', 'F', 0.5, 'small', CURRENT_DATE)
        RETURNING id""")).scalars().all()
print("Inserted animal ids:", new_ids)

Inserted animal ids: [3005, 3006, 3007]


In [12]:
run_to_pile(bronze_pipeline, build_bronze_resources(), write_disposition="append")

[silo] new rows loaded: {'postgres_animals': 3}


### The snapshots of the table

Every `append` created a new snapshot. `total_rows` grows, `schema_id` stays the same.

In [13]:
animals = catalog.load_table(f"{BRONZE}.postgres_animals")
snapshots(animals)

,snapshot_id,committed_at,operation,added_rows,deleted_rows,total_rows,schema_id
0,1520607404253395009,2026-11-06 10:59:15.861+00:00,append,3003,0,3003,0
1,6747709402664700157,2026-11-06 10:59:25.270+00:00,append,3,0,3006,0


### Read the table BEFORE and AFTER the change

`table.scan(snapshot_id=...)` reads the table **as it was** at that snapshot.

In [14]:
snapshot_after = animals.current_snapshot().snapshot_id
before_df = animals.scan(snapshot_id=snapshot_before).to_pandas()
after_df = animals.scan(snapshot_id=snapshot_after).to_pandas()

print(f"BEFORE (snapshot {snapshot_before}): {len(before_df)} rows")
print(f"AFTER  (snapshot {snapshot_after}): {len(after_df)} rows")
print("\nRows that exist only AFTER the change:")
after_df[~after_df.id.isin(before_df.id)][["id", "name", "species", "updated_at", "_load_id"]]

BEFORE (snapshot 1520607404253395009): 3003 rows
AFTER  (snapshot 6747709402664700157): 3006 rows

Rows that exist only AFTER the change:


,id,name,species,updated_at,_load_id
0,3005,Pile Test 1,dog,2026-11-06 10:59:21.246068,1793962761.8798635
1,3006,Pile Test 2,cat,2026-11-06 10:59:21.246068,1793962761.8798635
2,3007,Pile Test 3,rabbit,2026-11-06 10:59:21.246068,1793962761.8798635


In [15]:
# Time travel by TIME instead of snapshot id: "what did the table look like at this moment?"
first_snapshot = animals.snapshots()[0]
as_of = animals.snapshot_as_of_timestamp(first_snapshot.timestamp_ms)
print("Snapshot valid at", pd.to_datetime(first_snapshot.timestamp_ms, unit="ms", utc=True), "→", as_of.snapshot_id)
print("Rows at that time:", len(animals.scan(snapshot_id=as_of.snapshot_id).to_arrow()))

Snapshot valid at 2026-11-06 10:59:15.861000+00:00 → 1520607404253395009
Rows at that time: 3003


In [16]:
# The new snapshot only ADDED a small data file, the old one is untouched
data_files(animals)

,file_path,record_count,file_size_in_bytes
0,bronze/postgres_animals/data/ingestion_date=2026-11-06/00000-0-8f8ea49c-394c-4663-b5ec-3aafa32a7e54.parquet,3,7525
1,bronze/postgres_animals/data/ingestion_date=2026-11-06/00000-0-69013777-0004-4fdb-b163-a6ed6d6692db.parquet,3003,111127


## Bonus – incremental load into the additional pile (RustFS)

Same resources, a second pipeline (its own incremental state) and the **RustFS catalog**.

In [17]:
ensure_pile_bucket("rustfs")
bronze_pipeline_rustfs = dlt.pipeline(
    pipeline_name="pile_incremental_rustfs",
    destination=pile_destination("rustfs"),
    dataset_name=BRONZE,
    pipelines_dir=PIPELINES_DIR,
)
run_to_pile(bronze_pipeline_rustfs, build_bronze_resources(), write_disposition="append", pile="rustfs")
show_catalog_db("rustfs")

Bucket pawhome-pile already exists in rustfs


/opt/conda/lib/python3.13/site-packages/pyiceberg/io/__init__.py:380: UserWarning: No preferred file implementation for scheme: memory
  if file_io := _infer_file_io_from_scheme(location, properties):


[rustfs] new rows loaded: {'postgres_adopters': 2000, 'postgres_animals': 3006, 'postgres_applications': 5000, 'postgres_shelters': 50}


,table_namespace,table_name,metadata_location
0,bronze,postgres_adopters,s3://pawhome-pile/bronze/postgres_adopters/metadata/00001-7ca8f5d4-bb7d-4806-8793-eb189fc2fe34.metadata.json
1,bronze,postgres_animals,s3://pawhome-pile/bronze/postgres_animals/metadata/00001-ab02008c-d15d-4cf4-a0d3-cade0141b1c0.metadata.json
2,bronze,postgres_applications,s3://pawhome-pile/bronze/postgres_applications/metadata/00001-37842693-c35e-419a-ac13-0df8607feec8.metadata.json
3,bronze,postgres_shelters,s3://pawhome-pile/bronze/postgres_shelters/metadata/00001-fb71aed9-c1e8-4627-b867-78962fa70f58.metadata.json


In [18]:
# Rerun → nothing new; and the same row counts as in Silo
run_to_pile(bronze_pipeline_rustfs, build_bronze_resources(), write_disposition="append", pile="rustfs")
catalog_rustfs = get_catalog("rustfs")
pd.DataFrame([{"table": f"postgres_{n}",
               "rows_silo": len(catalog.load_table(f"{BRONZE}.postgres_{n}").scan().to_arrow()),
               "rows_rustfs": len(catalog_rustfs.load_table(f"{BRONZE}.postgres_{n}").scan().to_arrow())}
              for n in DB_TABLES])

[rustfs] new rows loaded: none


,table,rows_silo,rows_rustfs
0,postgres_shelters,50,50
1,postgres_adopters,2000,2000
2,postgres_animals,3006,3006
3,postgres_applications,5000,5000
